# Preparing the Data for Analysis

## Initialization

In [ ]:
# Import required libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.utils import shuffle

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix
)

In [ ]:
# Configure display settings for this analysis
pd.set_option('display.max_colwidth', None)

## Loading the Data

In [ ]:
# Load the dataset
df = pd.read_csv('datasets/Churn.csv')

## Initial Assessment

In [ ]:
# Inspect dataset structure
df.info()

In [ ]:
# Inspect a sample of records
print(df.sample(5))

In [ ]:
# Inspect initial descriptive statistics
print(df.describe())

**Initial dataset assessment**

The dataset contains 10,000 rows and 14 columns, including customer identifiers, demographic and financial attributes, and the `Exited` target. Initial inspection covers data types, descriptive statistics, missing values, and the class distribution. The `Tenure` column contains missing values that will require treatment before modeling. The row-number field is an administrative index rather than a predictive feature.

## Preprocessing

### Removing the `RowNumber` Column

In [ ]:
# Remove the RowNumber column
df = df.drop(['RowNumber'], axis=1)

In [ ]:
# Confirm RowNumber was removed
print(df.info())

### Standardizing Column Names

In [ ]:
# Inspect current column names
print('Nome das colunas:\n', df.columns)

In [ ]:
# Standardize column names
new_col_names = []
for old_name in df.columns:
    name_stripped = old_name.strip()
    name_lowered = name_stripped.lower()
    name_no_spaces = name_lowered.replace(' ', '_')
    new_col_names.append(name_no_spaces)

df.columns = new_col_names

print('Nome das colunas:\n', df.columns)

In [ ]:
# Rename columns using snake_case
df = df.rename(columns={
    'customerid': 'customer_id',
    'creditscore': 'credit_score',
    'numofproducts': 'num_of_products',
    'hascrcard': 'has_cr_card',
    'isactivemember': 'is_active_member',
    'estimatedsalary': 'estimated_salary'
})

print('Nome das colunas:\n', df.columns)

**Preprocessing summary**

The `RowNumber` column was removed because it duplicates the role of the DataFrame index. Column names were standardized to `snake_case` to make the analysis and subsequent feature processing more consistent.

## Data Preparation

### Assessing Missing Values

In [ ]:
# Check for missing values
print('Missing values by column:')
print(df.isna().sum())

### Examining Missing Values in `tenure`

In [ ]:
# Inspect tenure before deciding how to handle missing values
print('Descriptive statistics for tenure:')
df['tenure'].agg(['mean', 'median', 'min', 'max', 'skew'])


### Checking the Data Type of `tenure`

In [ ]:
# Inspect the tenure data type
df['tenure'].value_counts()

In [ ]:
# Convert tenure to nullable integer
df['tenure'] = df['tenure'].astype('Int64')

In [ ]:
# Confirm the tenure data type
print('tenure data type:', df['tenure'].dtype)

### Checking for Fully Duplicated Rows

In [ ]:
# Check for fully duplicated rows
print('Number of duplicated rows:', df.duplicated().sum())

### Checking for Duplicate Customers in `customer_id`

In [ ]:
# Check unique customer IDs
print('Unique customer_id values:',
      df['customer_id'].nunique())
print('Dataset row count:', df.shape[0])

**Data preparation summary**

**Missing values:** Missing observations were found in `tenure`. To avoid data leakage, imputation will be performed after splitting the dataset, using the median from the training set only.

**Data types:** The `tenure` column was converted to nullable integer format during the initial inspection.

**Duplicates:** The dataset was checked for fully duplicated records and repeated customer identifiers. These checks help establish the integrity of the observations before training.

# Training Models on Imbalanced Data

**Plan**

- Examine the target-class distribution.
- Define the features and target.
- Split the data reproducibly into training, validation, and test sets.
- Prepare features and compare baseline classification models.

## Evaluating Class Imbalance

In [ ]:
# Create a DataFrame to assess the exited class distribution
class_balance = pd.DataFrame({
    'count': df['exited'].value_counts(),
    'percentege': df['exited'].value_counts(normalize=True) * 100
}).sort_index()

display(class_balance.round(2))

In [ ]:
# Plot the target-class distribution

fig, ax = plt.subplots(figsize=(6,4))

bars = ax.bar(
    ['Permaneceram (0)', 'Saíram (1)'],
    class_balance['count'],
    color=['steelblue', 'darkorange']
)

ax.set(
    title='Customer churn: class distribution',
    xlabel='Saída de clientes',
    ylabel='Number of customers'
)
ax.set_ylim(0, class_balance['count'].max()*1.15)

plt.tight_layout()
plt.show()

## Defining Features and Target

In [ ]:
# Separate features from the target
target = df['exited']
features = df.drop(['exited', 'customer_id', 'surname'], axis=1)

## Creating Reproducible Data Splits

### First Split: Reserving 40% for Validation and Testing

In [ ]:
features_train, features_temp, target_train, target_temp = train_test_split(
    features, target, test_size=0.40, random_state=67, stratify=target
)

### Second Split: Dividing the Reserved 40% Equally

In [ ]:
features_valid, features_test, target_valid, target_test = train_test_split(
    features_temp, target_temp, test_size=0.50, random_state=67, stratify=target_temp
)

### Checking Split Sizes, Class Proportions, and Missing `tenure` Values

In [ ]:
for name, split_features, split_target in [
    ('Training', features_train, target_train),
    ('Validation', features_valid, target_valid),
    ('Test', features_test, target_test)
]:
    print(f'{name}: {len(split_features)} linhas |'
          f'Rotatividade de clientes: {split_target.mean():.2%} | '
          f'Missing values in tenure: {split_features["tenure"].isna().sum()}'
          )

## Preprocessing Data for the Models

### Handling Missing Values and Categorical Features

In [ ]:
# Calculate median tenure using the training set
tenure_median = features_train['tenure'].median()

In [ ]:
# Impute missing tenure values using the training median
# codificando as colunas categóricas com one-hot encoding

# dicionário para armazenar os conjuntos de features codificados
encoded_features = {}

for name, split_features in [
    ('training', features_train),
    ('validação', features_valid),
    ('teste', features_test)   
]:
    prepared_features = split_features.copy()

    # Impute missing tenure values using the training-set median
    prepared_features['tenure'] = (
        prepared_features['tenure'].astype('float64').fillna(tenure_median)
    )

    # One-hot encode the gender column
    encoded_gender = pd.get_dummies(
        prepared_features, columns=['gender'], drop_first=True,
        dtype=int
    )

    encoded = encoded_gender.copy()

    # One-hot encode the geography column
    encoded = pd.get_dummies(
        encoded_gender, columns=['geography'],
        dtype=int
    )

    # Align validation and test columns with training columns
    if name != 'training':
        encoded = encoded.reindex(
            columns=encoded_features['training'].columns,
            fill_value=0
        )

    # Store the encoded feature split in the dictionary
    encoded_features[name] = encoded


In [ ]:
# Assign encoded feature splits to separate variables
features_train_encoded = encoded_features['training']
features_valid_encoded = encoded_features['validação']
features_test_encoded = encoded_features['teste']

### Scaling Numerical Features for Logistic Regression

In [ ]:
# Select numerical columns
numeric_columns = [
    'credit_score',
    'age',
    'tenure',
    'balance',
    'num_of_products',
    'estimated_salary'
]

In [ ]:
# Fit scaling parameters on the training set
scaler = StandardScaler()
scaler.fit(features_train_encoded[numeric_columns])

In [ ]:
# Scale numerical features across encoded splits
scaled_features = {}

for name, split_features in [
    ('training', features_train_encoded),
    ('validação', features_valid_encoded),
    ('teste', features_test_encoded)
]:
    scaled = split_features.copy()

    scaled[numeric_columns] = scaler.transform(
        split_features[numeric_columns]
    )

    scaled_features[name] = scaled

In [ ]:
# Assign scaled feature splits to separate variables
features_train_scaled = scaled_features['training']
features_valid_scaled = scaled_features['validação']
features_test_scaled = scaled_features['teste']

In [ ]:
display(
    features_train_scaled[numeric_columns]
    .describe()
    .round(3)
)

## Training Baseline Models Without Class Balancing

### Training Logistic Regression on Scaled Features

In [ ]:
# Initialize the Logistic Regression baseline
logistic_model = LogisticRegression(
    random_state=67, 
    solver='liblinear',
    max_iter=1000
)

# Fit Logistic Regression on the training set
logistic_model.fit(features_train_scaled, target_train)

### Training Random Forest on Encoded, Unscaled Features

In [ ]:
forest_model = RandomForestClassifier(
    n_estimators=100,
    random_state=67,
    class_weight=None
)

forest_model.fit(features_train_encoded, target_train)

### Evaluating Both Models with the Same Metrics

In [ ]:
results = []

for name, model, validation_features in [
    ('Logistic Regression', logistic_model, features_valid_scaled),
    ('Random Forest', forest_model, features_valid_encoded)
]:
    predictions = model.predict(validation_features)
    probabilities = model.predict_proba(validation_features)[:, 1]

    results.append({
        'model': name,
        'accuracy': accuracy_score(target_valid, predictions),
        'precision': precision_score(target_valid, predictions, zero_division=0),
        'recall': recall_score(target_valid, predictions, zero_division=0),
        'f1': f1_score(target_valid, predictions, zero_division=0),
        'roc_auc': roc_auc_score(target_valid, probabilities)
    })

    matrix = pd.DataFrame(
        confusion_matrix(target_valid, predictions),
        index=['Actually Stayed', 'Actually Left'],
        columns=['Predicted Stayed', 'Predicted Left']
    )

    print('Matriz de Confusão do modelo de', name)
    display(matrix)

results_df = pd.DataFrame(results).set_index('model')

print('\nModel evaluation results on the validation set:')
display(results_df.round(3))

## Evaluating the Unbalanced Baseline Models

**Class distribution:** The target is imbalanced: 7,963 customers (79.63%) stayed, while 2,037 (20.37%) left. Accuracy alone can therefore be misleading.

**Feature preparation:** Customer identifiers and surnames were excluded from the predictors. The data was split into training (60%), validation (20%), and test (20%) subsets using stratification. Missing `tenure` values were imputed with the training-set median, categorical variables were one-hot encoded, and numerical features were scaled for Logistic Regression.

**Baseline comparison:** Logistic Regression produced a validation F1-score of **0.289** and ROC-AUC of **0.758**. Random Forest performed better, with an F1-score of **0.581** and ROC-AUC of **0.845**.

**Interpretation:** Neither baseline model exceeded the required F1-score of 0.59. Random Forest was the stronger starting point, but class-imbalance strategies and hyperparameter tuning were needed to improve the balance between precision and recall.

# Improving Model Performance

### Comparing Three Class-Balancing Strategies Against the Baseline

In [ ]:
results_balancing = []
trained_model = {}

for model_name, training_features,validation_features in [
    ('Logistic Regression', features_train_scaled, features_valid_scaled),
    ('Random Forest', features_train_encoded, features_valid_encoded)
]:
    for approach in [
        'Baseline',
        'Oversampling',
        'Undersampling',
        'Class weighting'
    ]:
        # Start each experiment with the original training set
        features_fit = training_features
        target_fit = target_train
        class_weight = None

        if approach == 'Oversampling':
            features_zeros = training_features[target_train == 0]
            features_ones = training_features[target_train == 1]
            target_zeros = target_train[target_train == 0]
            target_ones = target_train[target_train ==1]

            features_fit = pd.concat([features_zeros] + [features_ones] * round(len(features_zeros) / len(features_ones)))
            target_fit = pd.concat([target_zeros] + [target_ones] * round(len(target_zeros) / len(target_ones)))
            
            features_fit, target_fit = shuffle(features_fit, target_fit, random_state=67)
            
        elif approach == 'Undersampling':
            features_zeros = training_features[target_train == 0]
            features_ones = training_features[target_train == 1]
            target_zeros = target_train[target_train == 0]
            target_ones = target_train[target_train == 1]

            features_fit = pd.concat(
                [features_zeros.sample(
                    frac=(len(features_ones) / len(features_zeros)), random_state=67
                )] + [features_ones]
            )
            
            target_fit = pd.concat(
                [target_zeros.sample(
                    frac=(len(target_ones) / len(target_zeros)), random_state=67
                )] + [target_ones]
            )
                        
            features_fit, target_fit = shuffle(
                features_fit, target_fit, random_state=67
            )
                      
        elif approach == 'Class weighting':
            class_weight = 'balanced'

        # Initialize a fresh model for this experiment
        if model_name == 'Logistic Regression':
            model = LogisticRegression(
                solver='liblinear',
                max_iter=1000,
                random_state=67,
                class_weight=class_weight
            )
        else:
            model = RandomForestClassifier(
                n_estimators=100,
                random_state=67,
                class_weight=class_weight
            )

        model.fit(features_fit, target_fit)

        # Evaluate the trained model on validation data
        predictions = model.predict(validation_features)
        probabilities = model.predict_proba(validation_features)[:, 1]

        results_balancing.append({
            'model': model_name,
            'approach': approach,
            'training rows': len(target_fit),
            'accuracy': accuracy_score(target_valid, predictions),
            'precision': precision_score(target_valid, predictions, zero_division=0),
            'recall': recall_score(target_valid, predictions, zero_division=0),
            'f1': f1_score(target_valid, predictions, zero_division=0),
            'roc_auc': roc_auc_score(target_valid, probabilities)
        })
        
        trained_model[f'{model_name} - {approach}'] = model

## Comparing Class-Balancing Results

In [ ]:
balancing_results_df = pd.DataFrame(results_balancing)

display(
    balancing_results_df
    .sort_values('f1', ascending=False)
    .round(3)
)

### Finding the Best Hyperparameter Configuration

In [ ]:
# Prepare hyperparameter candidates
forest_params = []

for est in range(50, 101, 10):
    for depth in [5, 10, None]:
        for leaf in [1,3]:
            forest_params.append({
                'n_estimators': est,
                'max_depth': depth,
                'min_samples_leaf': leaf
            })

logistic_params = [
    {'C': 0.01, 'solver':'liblinear'},
    {'C': 0.1, 'solver':'liblinear'},
    {'C': 1, 'solver':'liblinear'},
    {'C': 10, 'solver':'liblinear'}
]

In [ ]:
# Initialize tracking variables
tuning_results = []

best_f1 = {
    'Logistic Regression': -1,
    'Random Forest': -1
}

best_models = {}
best_settings = {}

In [ ]:
# Extend the experiment with hyperparameter search
for model_name, training_features,validation_features in [
    ('Logistic Regression', features_train_scaled, features_valid_scaled),
    ('Random Forest', features_train_encoded, features_valid_encoded)
]:
    # Select candidate parameters for each model
    if model_name == 'Logistic Regression':
         params_options = logistic_params
    else:
         params_options = forest_params
    
    for approach in [
        'Baseline',
        'Oversampling',
        'Undersampling',
        'Class weighting'
    ]:
        # Start each experiment with the original training set
        features_fit = training_features
        target_fit = target_train
        class_weight = None

        if approach == 'Oversampling':
            features_zeros = training_features[target_train == 0]
            features_ones = training_features[target_train == 1]
            target_zeros = target_train[target_train == 0]
            target_ones = target_train[target_train ==1]
    
            features_fit = pd.concat([features_zeros] + [features_ones] * round(len(features_zeros) / len(features_ones)))
            target_fit = pd.concat([target_zeros] + [target_ones] * round(len(target_zeros) / len(target_ones)))
    
            features_fit, target_fit = shuffle(features_fit, target_fit, random_state=67)
    
        elif approach == 'Undersampling':
            features_zeros = training_features[target_train == 0]
            features_ones = training_features[target_train == 1]
            target_zeros = target_train[target_train == 0]
            target_ones = target_train[target_train == 1]
    
            features_fit = pd.concat(
                [features_zeros.sample(
                    frac=(len(features_ones) / len(features_zeros)), random_state=67
                )] + [features_ones]
            )
            target_fit = pd.concat(
                [target_zeros.sample(
                    frac=(len(target_ones) / len(target_zeros)), random_state=67
                )] + [target_ones]
            )
                        
            features_fit, target_fit = shuffle(
                features_fit, target_fit, random_state=67
            )
        
        elif approach == 'Class weighting':
            class_weight = 'balanced'

        # Initialize a fresh model for this experiment
        for params in params_options:
             if model_name == 'Logistic Regression':
                  model = LogisticRegression(
                       max_iter=1000,
                       class_weight=class_weight,
                       random_state=67,
                       **params
                  )
             else:
                model = RandomForestClassifier(
                     random_state=67,
                     class_weight=class_weight,
                     **params
                )

             model.fit(features_fit, target_fit)

             # Evaluate the trained model on validation data
             predictions = model.predict(validation_features)
             probabilities = model.predict_proba(validation_features)[:, 1]

             score = f1_score(target_valid, predictions, zero_division=0)

             result = {
                'model': model_name,
                'approach': approach,
                'parameters': params.copy(),
                'f1': score,
                'precision': precision_score(target_valid, predictions, zero_division=0),
                'recall': recall_score(target_valid, predictions, zero_division=0),
                'roc_auc': roc_auc_score(target_valid, probabilities)
             }

             tuning_results.append(result)
                        
             if score > best_f1[model_name]:
                best_f1[model_name] = score
                best_models[model_name] = model
                best_settings[model_name] = result.copy()


In [ ]:
# Inspect tuning results
tuning_results_df = pd.DataFrame(tuning_results)

display(
    tuning_results_df
    .sort_values('f1', ascending=False, kind='stable')
    .head(10)
    .round(3)
)

In [ ]:
# Display the best F1 result for each model
for model_name, settings in best_settings.items():
    print(f'\nBest {model_name}')
    print('Approach:', settings['approach'])
    print('Parameters', settings['parameters'])
    print(f'Validation F1: {settings["f1"]:.3f}')
    print(f'Validation ROC-AUC: {settings["roc_auc"]:.3f}')

### Assessing the Improvements

**Class-balancing approaches:** Three strategies were compared with the original training distribution: random oversampling, random undersampling, and class weighting. For Random Forest, balanced class weights improved validation F1-score from **0.581** to **0.634**. Recall increased from **0.479** to **0.627**, while precision decreased from **0.739** to **0.641**.

**Hyperparameter selection:** Logistic Regression did not reach the project's required F1-score above 0.59; its best validation F1-score was **0.497**. Several Random Forest configurations met the requirement. The highest validation F1-score was achieved by **Random Forest with balanced class weights**, using `n_estimators=100`, `max_depth=None`, and `min_samples_leaf=1`.

**Metric interpretation:** F1-score is the harmonic mean of precision and recall. Configurations with similar F1-scores can still have different precision–recall trade-offs, which matter for retention decisions.

**ROC-AUC:** The selected Random Forest achieved a validation ROC-AUC of **0.847**, substantially above the random-classifier reference of 0.5. ROC-AUC complements F1-score by measuring the model's ability to rank customers by churn risk across classification thresholds.

# Final Test Evaluation

In [ ]:
# Initialize the final model with selected parameters
final_model = RandomForestClassifier(
        random_state=67,
        class_weight='balanced',
        n_estimators=100,
        max_depth=None,
        min_samples_leaf=1
)

final_model.fit(features_train_encoded, target_train)

In [ ]:
# Generate predictions for the test set
test_predictions = final_model.predict(features_test_encoded)

test_probabilities = final_model.predict_proba(
    features_test_encoded
)[:, 1]

In [ ]:
# Calculate and display test metrics
test_f1 = f1_score(target_test, test_predictions, zero_division=0)

test_results = pd.DataFrame({
    'accuracy': [accuracy_score(target_test, test_predictions)],
    'precision': [precision_score(target_test, test_predictions, zero_division=0)],
    'recall': [recall_score(target_test, test_predictions, zero_division=0)],
    'f1': [test_f1],
    'roc_auc': [roc_auc_score(target_test, test_probabilities)]
}, index=['Final random forest'])

display(test_results.round(3))

In [ ]:
# Build the confusion matrix
test_matrix = pd.DataFrame(
    confusion_matrix(
        target_test,
        test_predictions,
        labels=[0, 1]
    ),
    index=['Actually Stayed', 'Actually Left'],
    columns=['Predicted Stayed', 'Predicted Left']
)

display(test_matrix)

## Overall Conclusion

The objective of this project was to develop a binary classification model to identify bank customers at higher risk of leaving, achieving an F1-score above **0.59**.

**Data preparation and exploration**

The dataset contains 10,000 customer records, of which approximately **20.37%** belong to the churn class. This imbalance makes it important to evaluate models using metrics beyond accuracy.

The workflow included data cleaning, column-name standardization, missing-value treatment, one-hot encoding, and numerical feature scaling where appropriate. The dataset was split using stratification into training (60%), validation (20%), and test (20%) subsets, preserving the target distribution. Undersampling was implemented using pandas to reduce the majority class to approximately the size of the minority class.

**Model comparison and optimization**

Two algorithms were evaluated: Logistic Regression and Random Forest. Without class-imbalance treatment, Random Forest performed better, achieving a validation F1-score of **0.581**, compared with **0.289** for Logistic Regression.

Three imbalance-handling strategies were compared: oversampling, undersampling, and class weighting. A hyperparameter search was then conducted for both models. Logistic Regression improved but reached a best validation F1-score of only **0.497** (balanced class weights, `C=0.1`). Random Forest exceeded the required threshold in several configurations. The selected model used balanced class weights, `n_estimators=100`, `max_depth=None`, and `min_samples_leaf=1`, achieving validation F1-score **0.634**, precision **0.641**, recall **0.627**, and ROC-AUC **0.847**.

**Final test results**

The final model was a Random Forest classifier with `class_weight='balanced'`, `n_estimators=100`, `max_depth=None`, and `min_samples_leaf=1`. On the held-out test set, it achieved accuracy **0.849**, precision **0.635**, recall **0.613**, F1-score **0.623**, and ROC-AUC **0.869**.

The confusion matrix shows that the model correctly identified **250 of the 408 customers who left**. It missed **158** churners and incorrectly flagged **144** customers who stayed.

The test F1-score (**0.623**) was slightly lower than the validation F1-score (**0.634**). The configuration was selected based on validation performance among multiple candidates, so a modest difference on unseen test data is unsurprising.

**Business implications**

The project achieved its objective: test F1-score **0.623** exceeds the minimum requirement of **0.59**. The model identifies approximately **61.3%** of customers who leave, while producing both false positives and false negatives.

This analysis demonstrates why class imbalance and multiple evaluation metrics matter in churn prediction. The model could support early identification of at-risk customers and help prioritize retention activities.

Before operational use, the bank should assess the business costs of false positives and false negatives, optimize the classification threshold according to retention priorities, and validate performance on newer, representative data.

**Reproducibility**

The original notebook reports Python **3.12.10**. Package versions and results may vary between environments; verify the installed scikit-learn version before citing it as an exact reproduction requirement.